In [1]:
!pip install -U spacy
!pip install -U scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz -q

  Using cached spacy-3.8.15-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (28 kB)
Using cached spacy-3.8.15-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (32.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 37.8 MB/s eta 0:00:00
  Attempting uninstall: confection
    Found existing installation: confection 0.1.5
    Uninstalling confection-0.1.5:
      Successfully uninstalled confection-0.1.5
  Attempting uninstall: blis
    Found existing installation: blis 0.7.11
    Uninstalling blis-0.7.11:
      Successfully uninstalled blis-0.7.11
  Attempting uninstall: thinc
    Found existing installation: thinc 8.2.5
    Uninstalling thinc-8.2.5:
      Successfully uninstalled thinc-8.2.5
  Attempting uninstall: weasel
    Found existing installation: weasel 0.4.3
    Uninstalling weasel-0

In [2]:
import os
import json
import pickle
import time
import sys

import pandas as pd
import numpy as np
from tqdm import tqdm

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

sys.path.append(SRC_PATH)

print("SRC path added:", SRC_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SRC path added: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/src


In [3]:
ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local"

BENCHMARK_PATH = f"{ROOT}/benchmarks/aria_local_benchmark_v2.json"

GRAPH_PATH = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/aria_lite_graph_v2_1_communities_v1.pkl"

OUTPUT_DIR = f"{ROOT}/failure_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
with open(GRAPH_PATH, "rb") as f:
    graph = pickle.load(f)

print("Nodes :", graph.number_of_nodes())
print("Edges :", graph.number_of_edges())

Nodes : 2105
Edges : 22506


In [5]:
with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)["queries"]

print(len(benchmark), "queries loaded")

10 queries loaded


In [6]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

sys.path.append(PROJECT_ROOT)

from local_traversal import run_local_query

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [7]:
analysis_rows = []

for item in tqdm(benchmark):

    query = item["query"]
    query_id = item["id"]

    gold_ids = {
        x["node_id"]
        for x in item["gold_evidence"]
    }

    start = time.time()

    retrieved = run_local_query(graph, query)

    latency = time.time() - start

    retrieved_ids = [
        r["node_id"]
        for r in retrieved
    ]

    retrieved_set = set(retrieved_ids)

    ####################################################
    # Candidate statistics
    ####################################################

    candidate_count = len(retrieved_ids)

    gold_retrieved = gold_ids.intersection(retrieved_set)

    gold_retrieved_count = len(gold_retrieved)

    gold_coverage = (
        gold_retrieved_count / len(gold_ids)
        if len(gold_ids) > 0
        else 0
    )

    ####################################################
    # Ranking statistics
    ####################################################

    gold_ranks = []

    for gid in gold_ids:

        if gid in retrieved_ids:

            rank = retrieved_ids.index(gid) + 1
            gold_ranks.append(rank)

    first_gold_rank = (
        min(gold_ranks)
        if len(gold_ranks) > 0
        else None
    )

    last_gold_rank = (
        max(gold_ranks)
        if len(gold_ranks) > 0
        else None
    )

    ####################################################
    # Failure category
    ####################################################

    if candidate_count == 0:

        failure = "No retrieval"

    elif gold_retrieved_count == 0:

        failure = "Traversal failure"

    elif first_gold_rank > 10:

        failure = "Ranking failure"

    elif candidate_count > 500:

        failure = "Candidate explosion"

    else:

        failure = "Good retrieval"

    ####################################################

    analysis_rows.append({

        "query_id": query_id,

        "anchor_entity": item["anchor_entity"],

        "candidate_count": candidate_count,

        "gold_count": len(gold_ids),

        "gold_retrieved": gold_retrieved_count,

        "gold_coverage": gold_coverage,

        "first_gold_rank": first_gold_rank,

        "last_gold_rank": last_gold_rank,

        "latency_sec": latency,

        "failure_category": failure

    })

  0%|          | 0/10 [00:00<?, ?it/s]


KeyError: 'node_id'

In [ ]:
failure_df = pd.DataFrame(analysis_rows)

failure_df

In [ ]:
failure_df["failure_category"].value_counts()

In [ ]:
failure_df.sort_values(
    "candidate_count",
    ascending=False
)[[
    "query_id",
    "anchor_entity",
    "candidate_count",
    "gold_retrieved",
    "failure_category"
]]

In [ ]:
failure_df.sort_values(
    "first_gold_rank"
)[[
    "query_id",
    "anchor_entity",
    "first_gold_rank",
    "last_gold_rank",
    "gold_coverage"
]]

In [ ]:
summary = {

    "mean_candidates":
        failure_df.candidate_count.mean(),

    "median_candidates":
        failure_df.candidate_count.median(),

    "mean_gold_coverage":
        failure_df.gold_coverage.mean(),

    "median_first_gold_rank":
        failure_df.first_gold_rank.median(),

    "mean_latency":
        failure_df.latency_sec.mean()

}

summary

In [ ]:
failure_df.to_csv(
    f"{OUTPUT_DIR}/failure_analysis.csv",
    index=False
)

with open(
    f"{OUTPUT_DIR}/failure_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print("Saved.")